# B05 · Sesión 2 — Un motor de reglas propio y DMN

**Objetivo (RA5-a):** construir un motor *match-resolve-act* en Python puro, sin dependencias, y traducir una política de negocio a una **tabla DMN**.

> Práctica guiada de la Sesión 2 de los [apuntes](../apuntes.md).

In [ ]:
from dataclasses import dataclass

@dataclass
class Regla:
    nombre: str
    prioridad: int
    condicion: object
    accion: object

class MotorReglas:
    def __init__(self):
        self.reglas, self.hechos, self.traza = [], {}, []

    def add(self, regla):
        self.reglas.append(regla)

    def declarar(self, **hechos):
        self.hechos.update(hechos)

    def run(self):
        while True:
            activas = [r for r in self.reglas if r.condicion(self.hechos) and r.nombre not in self.traza]
            if not activas:
                break
            r = max(activas, key=lambda x: x.prioridad)   # resolución de conflictos
            self.traza.append(r.nombre)
            r.accion(self.hechos)
        return self.hechos, self.traza

In [ ]:
m = MotorReglas()
m.add(Regla("critica", 30,
            lambda h: h.get("impacto") == "alto" or h.get("usuarios", 0) > 50,
            lambda h: h.update(nivel="critica")))
m.add(Regla("media", 20,
            lambda h: h.get("nivel") is None and h.get("usuarios", 0) > 10,
            lambda h: h.update(nivel="media")))
m.add(Regla("baja", 10,
            lambda h: h.get("nivel") is None,
            lambda h: h.update(nivel="baja")))

m.declarar(impacto="bajo", usuarios=30)
print(m.run())
# -> ({'impacto': 'bajo', 'usuarios': 30, 'nivel': 'media'}, ['media'])

## Actividad — política de devoluciones

Traduce esta política a reglas con `MotorReglas` (y luego a una tabla DMN en Markdown):

- Se **aprueba** si han pasado **menos de 30 días** y el producto está **sin usar**.
- Requiere **supervisor** si han pasado **entre 30 y 60 días**.
- Se **rechaza** si han pasado **más de 60 días**.

In [ ]:
# TODO: define las reglas de devolución y pruébalas con 3 casos
m = MotorReglas()
...

### Tabla DMN

| Días desde la compra | Estado del producto | Decisión |
|---|---|---|
| `< 30` | sin usar | Aprobar |
| `< 30` | usado | Revisar |
| `30–60` | cualquiera | Supervisor |
| `> 60` | cualquiera | Rechazar |

**Para casa:** ¿qué ventaja tiene separar las reglas del código (como en Drools/DMN) en un equipo real?